In [ ]:
# =========================
# Librairies
# =========================
from neo4j import GraphDatabase
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Connexion Neo4j
# =========================
uri = "bolt://localhost:7687"
user = "neo4j"
password = "plbconsultant"

driver = GraphDatabase.driver(uri, auth=(user, password))

# =========================
# Récupération des données clients
# =========================
query = """
MATCH (c:Customer)-[:PURCHASED]->(o:Order)
RETURN id(c) AS customer_id,
       c.companyName AS company,
       COUNT(o) AS num_orders,
       SUM(toFloat(o.freight)) AS total_freight
"""

with driver.session() as session:
    result = session.run(query)
    df = pd.DataFrame([dict(record) for record in result])

print("Aperçu des données récupérées :")
display(df.head())

# =========================
# Clustering KMeans
# =========================
# Features pour clustering : nombre de commandes et total du fret
X = df[['num_orders', 'total_freight']].fillna(0)

# Nombre de clusters souhaités
k = 4
kmeans = KMeans(n_clusters=k, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

print("Centres des clusters :")
print(kmeans.cluster_centers_)

# =========================
# Visualisation
# =========================
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='num_orders', y='total_freight', hue='cluster', palette='Set2', s=100)
plt.title("Segmentation des clients Northwind (KMeans)")
plt.xlabel("Nombre de commandes")
plt.ylabel("Total du fret")
plt.show()

# =========================
# Résultat final
# =========================
print("Clients avec cluster assigné :")
display(df.head(10))
